# 02 — Noise channels: Kraus ops, Bloch ball, fidelity, Qiskit cross-check

This is the Phase 2 demo. We build the standard 1-qubit channels (bit-flip, phase-flip, depolarizing, amplitude damping) in Kraus form, look at how each one deforms the Bloch ball, plot fidelity vs error rate, and cross-check our depolarizing channel against `qiskit_aer`.

Read alongside `notes/04-noise-channels.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qec import channels as ch
from qec import statevec as sv

ATOL = 1e-12
plt.rcParams["figure.figsize"] = (6, 4)


## 1. Kraus operators are CPTP

Every channel must satisfy `sum_k K_k^dag K_k = I`. Quick check:


In [ ]:
for name, K in {
    "bit_flip(0.3)":      ch.bit_flip(0.3),
    "phase_flip(0.3)":    ch.phase_flip(0.3),
    "depolarizing(0.4)":  ch.depolarizing(0.4),
    "amplitude_damping(0.5)": ch.amplitude_damping(0.5),
}.items():
    print(f"{name:30s} CPTP={ch.is_cptp(K)}")


## 2. Bloch-ball action

For a single qubit, every density matrix corresponds to a point in the Bloch ball:
`rho = (I + r_x X + r_y Y + r_z Z) / 2`. We sweep a grid of pure states on the Bloch sphere, push each one through a channel, and plot the resulting Bloch vectors.

Expectation:
- **depolarizing** shrinks the sphere uniformly (it's a "scalar" map),
- **phase-flip** flattens the equator (X and Y axes shrink, Z untouched),
- **amplitude damping** drags the sphere toward the north pole `|0>`.


In [ ]:
def bloch_vector(rho):
    rx = np.real(np.trace(sv.X @ rho))
    ry = np.real(np.trace(sv.Y @ rho))
    rz = np.real(np.trace(sv.Z @ rho))
    return np.array([rx, ry, rz])

def bloch_state(theta, phi):
    return np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)], dtype=complex)

# Grid on the Bloch sphere
thetas = np.linspace(0, np.pi, 18)
phis = np.linspace(0, 2*np.pi, 36)
sphere_pts = np.array([bloch_vector(ch.density_from_state(bloch_state(t, p)))
                       for t in thetas for p in phis])

def apply_to_grid(kraus):
    out = []
    for t in thetas:
        for p in phis:
            rho = ch.density_from_state(bloch_state(t, p))
            rho = ch.apply_channel_full(rho, kraus)
            out.append(bloch_vector(rho))
    return np.array(out)

panels = [
    ("input (Bloch sphere)", sphere_pts),
    ("depolarizing(0.5)",    apply_to_grid(ch.depolarizing(0.5))),
    ("phase_flip(0.5)",      apply_to_grid(ch.phase_flip(0.5))),
    ("amplitude_damping(0.5)", apply_to_grid(ch.amplitude_damping(0.5))),
]

fig = plt.figure(figsize=(13, 3.2))
for i, (title, pts) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 4, i, projection="3d")
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=4)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_box_aspect([1, 1, 1])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.tight_layout()
plt.show()


## 3. Fidelity vs error rate

For depolarizing noise on a pure input, the textbook result is
`F(rho_out, |psi><psi|) = 1 - p/2`. We plot fidelity vs `p` for several channels on input `|+>`. The depolarizing curve should land exactly on the analytic line.


In [ ]:
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
rho_in = ch.density_from_state(plus)
ps = np.linspace(0, 1, 41)

def fidelity_curve(channel_factory):
    return [ch.fidelity(ch.apply_channel_full(rho_in, channel_factory(p)), rho_in) for p in ps]

curves = {
    "bit_flip":          fidelity_curve(ch.bit_flip),
    "phase_flip":        fidelity_curve(ch.phase_flip),
    "depolarizing":      fidelity_curve(ch.depolarizing),
    "amplitude_damping": fidelity_curve(ch.amplitude_damping),
}

plt.figure()
for name, ys in curves.items():
    plt.plot(ps, ys, label=name)
plt.plot(ps, 1 - ps/2, "k--", label="1 - p/2 (analytic, depolarizing)")
plt.xlabel("error rate p (or gamma)")
plt.ylabel("F(rho_out, |+><+|)")
plt.title("fidelity of post-channel state with input |+>")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# numerical check that depolarizing matches the analytic line.
# Tolerance ~1e-7: ch.fidelity uses an eigendecomposition that is not
# bit-exact on intermediate p; the analytic relation still holds.
err = max(abs(F - (1 - p/2)) for p, F in zip(ps, curves["depolarizing"]))
print(f"max |F_depolarizing(p) - (1 - p/2)| = {err:.2e}")
assert err < 1e-7, "depolarizing fidelity should match 1 - p/2 within eigh precision"
print("OK")


## 4. apply_1q_channel agrees with explicit Kronecker

Phase 2 introduces multi-qubit density matrices. `qec.channels.apply_1q_channel` uses the same axis-reshape trick as `qec.statevec`, but on both row and column axes. We sanity-check by comparing against the explicit `I ⊗ ... ⊗ K ⊗ ... ⊗ I` construction on a 3-qubit GHZ state.


In [ ]:
def kron_apply(rho, kraus_1q, q, n):
    out = np.zeros_like(rho)
    for K in kraus_1q:
        ops = [np.eye(2, dtype=complex)] * n
        ops[n - 1 - q] = K
        M = ops[0]
        for op in ops[1:]:
            M = np.kron(M, op)
        out = out + M @ rho @ M.conj().T
    return out

n = 3
psi = sv.zero_state(n)
psi = sv.apply_1q(psi, sv.H, 0)
for q in range(1, n):
    psi = sv.cnot(psi, control=0, target=q)
rho = ch.density_from_state(psi)

K = ch.depolarizing(0.3)
for q in range(n):
    out_reshape = ch.apply_1q_channel(rho, K, q)
    out_kron    = kron_apply(rho, K, q, n)
    print(f"qubit {q}: max diff = {np.max(np.abs(out_reshape - out_kron)):.2e}")
    assert np.allclose(out_reshape, out_kron, atol=ATOL)
print("OK")


## 5. Cross-check: our depolarizing vs `qiskit_aer.noise`

`qiskit_aer.noise.depolarizing_error(p, 1)` builds the same channel; we extract its Kraus operators via `qiskit.quantum_info.Kraus(err)` and compare on a random input state.


In [ ]:
try:
    from qiskit.quantum_info import Kraus
    from qiskit_aer.noise import depolarizing_error
except ImportError as e:
    print("Skipping Qiskit cross-check (qiskit / qiskit-aer not installed):", e)
else:
    rng = np.random.default_rng(2026)
    psi = rng.normal(size=2) + 1j * rng.normal(size=2)
    psi /= np.linalg.norm(psi)
    rho_in = ch.density_from_state(psi)

    for p in [0.0, 0.1, 0.5, 0.9, 1.0]:
        ours = ch.apply_channel_full(rho_in, ch.depolarizing(p))

        err_op = depolarizing_error(p, 1)
        qk_kraus = [np.asarray(K, dtype=complex) for K in Kraus(err_op).data]
        theirs = sum(K @ rho_in @ K.conj().T for K in qk_kraus)

        diff = np.max(np.abs(ours - theirs))
        print(f"p = {p:.2f}: max abs diff = {diff:.2e}")
        assert np.allclose(ours, theirs, atol=1e-10)
    print("OK")


## What this notebook gates

- Kraus ops are CPTP for every parameter we'll use.
- The Bloch-ball pictures match the textbook intuition (uniform shrinkage / equator squash / drag-to-pole).
- Depolarizing fidelity is exactly `1 - p/2`.
- The reshape-based `apply_1q_channel` agrees with explicit Kronecker construction on a 3-qubit entangled state.
- Our depolarizing channel agrees with `qiskit_aer.noise.depolarizing_error` to numerical precision.

Next: Phase 3 — the stabilizer formalism. We'll build a tableau simulator (Aaronson-Gottesman) and use it as a faster, exact alternative to density-matrix evolution for Pauli noise.
